In [ ]:
import os
import pickle
import numpy as np
import librosa
from mutagen.easyid3 import EasyID3
from mutagen.mp3 import MP3

In [ ]:
# Charger le modèle
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

In [ ]:
def extract_features(filepath):
    audio, sr = librosa.load(filepath, duration=30)
    mfcc = np.mean(librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13), axis=1)
    chroma = np.mean(librosa.feature.chroma_stft(y=audio, sr=sr), axis=1)
    contrast = np.mean(librosa.feature.spectral_contrast(y=audio, sr=sr), axis=1)
    tempo, _ = librosa.beat.beat_track(y=audio, sr=sr)
    return np.hstack([mfcc, chroma, contrast, tempo])

In [ ]:
def sanitize(text):
    forbidden = r'\/:*?"<>|'
    for char in forbidden:
        text = text.replace(char, "")
    return text.strip()

In [ ]:
def rename_file(folder_path, filename):
    filepath = os.path.join(folder_path, filename)
    try:
        tags = EasyID3(filepath)
        title = tags.get("title", [None])[0]
        artist = tags.get("artist", [None])[0]

        if not title or not artist:
            print(f"⚠️  Métadonnées manquantes, skip : {filename}")
            return

        new_name = f"{sanitize(title)} - {sanitize(artist)}.mp3"
        new_path = os.path.join(folder_path, new_name)

        if os.path.exists(new_path) and new_path != filepath:
            print(f"⚠️  Fichier existant, skip : {new_name}")
        else:
            os.rename(filepath, new_path)
            print(f"✅ Renommé : {new_name}")

    except Exception as e:
        print(f"❌ Erreur sur {filename} : {e}")

In [ ]:
def rename_files(folder_path):
    mp3_files = [f for f in os.listdir(folder_path) if f.endswith(".mp3")]
    print(f"🗂️  Renommage de {len(mp3_files)} fichiers\n")

    for filename in mp3_files:
        rename_file(folder_path, filename)
        

In [ ]:
def predict_genre(folder_path, filename):
    filepath = os.path.join(folder_path, filename)
    try:
        tags = EasyID3(filepath)
        existing_genre = tags.get("genre", [None])[0]

        if existing_genre and existing_genre.lower() != "music":
            print(f"⏭️  Genre déjà renseigné ({existing_genre}), skip : {filename}")
            return

        features = extract_features(filepath)
        genre = model.predict([features])[0]
        tags["genre"] = genre
        tags.save()
        print(f"✅ Genre prédit : {genre} — {filename}")

    except Exception as e:
        print(f"❌ Erreur sur {filename} : {e}")

In [ ]:
def predict_genres(folder_path):
    mp3_files = [f for f in os.listdir(folder_path) if f.endswith(".mp3")]
    print(f"🎵 Prédiction de genre pour {len(mp3_files)} fichiers\n")

    for filename in mp3_files:
        predict_genre(folder_path, filename)

In [ ]:
folder = r"C:\Users\robin\Music\New Music"

#rename_files(folder)
predict_genres(folder)

In [ ]:
folder = r"C:\Users\robin\Music\New Music"
filename = r"School's Out - Lupus Nocte.mp3"

rename_file(folder, filename)

In [ ]:
# Stats sur les genres de ma musique personnelle
from mutagen.easyid3 import EasyID3
from collections import Counter
import os

folder_path = r"C:\Users\robin\Music\Musique"

genres = []
for filename in os.listdir(folder_path):
    if filename.endswith(".mp3"):
        try:
            tags = EasyID3(os.path.join(folder_path, filename))
            genre = tags.get("genre", [None])[0]
            if genre:
                genres.append(genre)
        except:
            pass

counter = Counter(genres)
for genre, count in counter.most_common():
    print(f"{genre} : {count}")